# Lab 02 - Data Visualization In Python

An important step in data analysis and machine learning is visualization. Visualization both of the data itself and visualization of the models and training processes. This gives insights that are hard to achieve otherwise.

For the purpose to the labs in the course we will use the [Plotly](https://plotly.com/python/) graphics library. In the following lab we will practice the creation of some basic graphs. you are encouraged to have a deeper look on the [Plotly-Python webiste](https://plotly.com/python/).

For this end, we will use two datasets:
- The first one, [based on this dataset](https://www.kaggle.com/spscientist/students-performance-in-exams) presents results of students in exams in different disciplines. In addition to scores to the different exams, the gender of the student is indicated as a categorical column, as his ethnicity, his parental level of education and whether the student was prepared with a course to the tests.

- The second is of [temperature measurements taken major cities](https://www.kaggle.com/sudalairajkumar/daily-temperature-of-major-cities)  from various countries (in Fahrenheit), in different days, months and years.

In [1]:
import sys
sys.path.append("../")
from utils import *

The following imports from the Plotly library are already imported in the `utils` file. We are importing them here for deductive reasons.

In [2]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

## Loading Datasets

In [15]:
students_df = pd.read_csv("../data/Students_Performance.csv")
students_df.head()

,Unnamed: 0,gender,race.ethnicity,parental.level.of.education,lunch,test.preparation.course,math.score,reading.score,writing.score,science.score
0,1,female,group B,bachelor's degree,standard,none,88.8,86.506024,88.444444,84.438076
1,2,female,group C,some college,standard,completed,87.6,95.180723,94.666667,62.911799
2,3,female,group B,master's degree,standard,none,96.0,97.590361,96.888889,58.740116
3,4,male,group A,associate's degree,free/reduced,none,78.8,79.277108,75.111111,79.261140
4,5,male,group C,some college,standard,none,90.4,89.397590,88.888889,87.833540


In [16]:
temperature_df = pd.read_csv("../data/City_Temperature.csv")
temperature_df = temperature_df.loc[temperature_df["AvgTemperature"] != -99]
temperature_df.head()

,Region,Country,City,Month,Day,Year,AvgTemperature
0,Africa,Algeria,Algiers,1,1,1995,64.2
1,Africa,Algeria,Algiers,1,2,1995,49.4
2,Africa,Algeria,Algiers,1,3,1995,48.8
3,Africa,Algeria,Algiers,1,4,1995,46.4
4,Africa,Algeria,Algiers,1,5,1995,47.9


## Distribution Of Features

When you first look at a new dataset, it is important to understand how the different features "behave". Generally, we distinguish between 3 kinds of features: categorical (i.e. gender, country, etc.), discrete (i.e. years in range 1950 to 1960) or continuous (i.e. price, weight, height, etc.). We will explore different visualize options for different features, both on themselves and together with other features.

### Categorical Features

One of the simplest and most used plots to visualize categorical or discrete data is a bar plot. We use it to visualize how many items are there for each category of a given feature.

We use the `px.bar` function of Plotly to do so. The 2 first parameters are `x` and `y`, where `x` spepcifies the different categories and `y` specifies the number of occurences of each of those categories. Thus, we need to build a `Pandas.DataFrame` object that fits this structure.

In [36]:
df_count_ethnicities = students_df.groupby(['race.ethnicity']).size().reset_index(name='Count')
df_count_ethnicities

,race.ethnicity,Count
0,group A,89
1,group B,190
2,group C,319
3,group D,262
4,group E,140


In [38]:
px.bar(df_count_ethnicities, x="race.ethnicity", y="Count", height=200).show()

To know whether the proportions of men and women are equal over the ethnicities, let's color by gender.

In [39]:
df_count_ethnicities = students_df.groupby(['race.ethnicity', 'gender']).size().reset_index(name='Count')
px.bar(df_count_ethnicities, x="race.ethnicity", y="Count", color = "gender", height=200).show()

Now, let us check if the parental educational degree influences whether the students had a test preparation course. to keepThis time, we keep it simple and make independent plots for each level of education, and we will next use another feature of Plotly to plot them together. 

In [34]:
colored_by = "test.preparation.course"
split_by = 'parental.level.of.education'

for level in students_df[split_by].unique():
  df = students_df.loc[students_df[split_by] == level]\
        .groupby([colored_by]).size().reset_index(name='Count')
  px.pie(df, values='Count', names = colored_by, title = level, height=150).show()

## Continuous features

Next, how should we deal with features with some continuous scale? Suppose we want to know what is the distribution of grades, across three differnt subjects.

We can do so py plotting a histogram of grades for each subject. We will look both at the absolute- and relative counts.
- Observe that for all three subjects we are getting a noisy bell like shape.
- Most students achieved grades around 85.
- Some students achieved very high grades, while other achieved lower grades of around 60-70.

Let us hope that the grades distribution of our course will be at least as good as these grades.


<br>Note, that unlike the separate figures used for the Pie charts, here we are using Plotly's `make_subplots` function to create all plots in a single figure.



In [32]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("Math score distribution", "Writing score distribution", "Reading score distribution"))

for i, label in enumerate(["math.score", "writing.score", "reading.score"]):
  fig.append_trace(go.Histogram(x=students_df[label], showlegend = False), row = 1, col = i+1)
  fig.update_xaxes(title_text=label.capitalize(), row=1, col=i+1)

fig.update_layout(height=300).show()

Often we want to know how two different continuous feature influence one another. It could be interesting to see if they correlate. Here we will check if there is a correlation between a student's grade in math and in reading or science. We do so by plotting a scatter plot with the `x` values being the one feature and the `y` values the other feature. We can also color the dots by some category like gender or some continuous value like the score.

In the figure below, we clearly see that the score at the math exam is highly correlated with the reading exam. For the science's exam, it's a little bit more complex: for the girls (in blue), the more they have a good note in math, the less they get in science.

In [71]:
students_df["gender.cat"] = pd.Categorical(students_df["gender"]).codes

fig = make_subplots(rows=1, cols=2, start_cell="bottom-left")

fig.add_traces([go.Scatter(x=students_df["math.score"], y=students_df["reading.score"], mode="markers", 
                           marker = dict(color = students_df["gender.cat"], colorscale="Bluered")),
                go.Scatter(x=students_df["math.score"], y=students_df["science.score"], mode="markers", 
                           marker = dict(color = students_df["gender.cat"], colorscale="Bluered"))],
               rows=[1,1], cols=[1,2])

fig.update_xaxes(title_text="Reading Score", row=1, col=1)
fig.update_xaxes(title_text="Science Score", row=1, col=2)
fig.update_yaxes(title_text="Math Score")
fig.update_traces(showlegend=True)
fig.show()


## Visualizing Combinations Of Features

When we have 2 categorical features and we want to know the distribution of samples across those 2 features, we often use heatmaps. A heatmap is also useful to represent any continuous feature across different catgorical features. 

In [80]:
df_count_ethnicities = students_df.groupby(['race.ethnicity', 'gender']).size().reset_index(name='Count')

x_ = np.unique(df_count_ethnicities["race.ethnicity"].tolist())
y_ = np.unique(df_count_ethnicities["gender"].tolist())

values = np.array(df_count_ethnicities["Count"]).reshape(5, 2)
values_norm_row = (values.T/values.sum(axis = 1)).T
values_norm_col = values/values.sum(axis = 0)

for title, z in [["Counts Heatmap", values], ["Row Normalized", values_norm_row], ["Column Normalized", values_norm_col]]:
    go.Figure(go.Heatmap(x=y_, y=x_,z=z), layout=go.Layout(title=title, height=300, width=200)).show()

In order to visualize the distribution of continuous features, we can use boxplots.

In the graph below, we display the distribution of the temperature of Paris and Bordeaux for each graph separately.
You firset observe see that summer's months (6-8) are warmer than the other months.
We also can compare for each month Bordeaux and Paris and globally see that Bordeaux is warmer than Paris. To see it a little easily, we can make line charts.

In [81]:
france_temperature = temperature_df[temperature_df['Country'] == "France"]
france_temperature["Month"] = pd.Categorical(france_temperature["Month"])
di = {1: "Jan", 2: "Feb", 3: "March", 4: "Apr", 5: "May", 6: "June", 7: "July",
      8: "Aug", 9 : "Sept", 10: "Oct", 11: "Nov", 12: "Dec"}
france_temperature["Month_lab"] = france_temperature["Month"] 
france_temperature.replace({"Month_lab": di})

C:\Users\gilad\anaconda3\envs\iml.env\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\gilad\anaconda3\envs\iml.env\lib\site-packages\ipykernel_launcher.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Region,Country,City,Month,Day,Year,AvgTemperature,Month_lab
712420,Europe,France,Paris,1,1,1995,37.4,Jan
712421,Europe,France,Paris,1,2,1995,35.2,Jan
712422,Europe,France,Paris,1,3,1995,36.0,Jan
712423,Europe,France,Paris,1,4,1995,30.7,Jan
712424,Europe,France,Paris,1,5,1995,27.0,Jan
...,...,...,...,...,...,...,...,...
730947,Europe,France,Bordeaux,5,9,2020,61.5,May
730948,Europe,France,Bordeaux,5,10,2020,56.4,May
730949,Europe,France,Bordeaux,5,11,2020,48.9,May
730950,Europe,France,Bordeaux,5,12,2020,50.2,May


In [82]:
px.scatter(france_temperature, x="Month_lab", y="AvgTemperature", color="City", facet_col = "City").show()

In [83]:
px.box(france_temperature, x="Month", y="AvgTemperature", color="City").show()

In [87]:
fig = make_subplots(rows=2, cols=1, start_cell="top-left")

fig.add_trace(go.Scatter(x=y_, y=x_,z=values_norm_all, zmin = 0, zmax = 1),
              row=1, col=1)

ValueError: Invalid properties specified for object of type plotly.graph_objs.Scatter: ('z', 'zmin', 'zmax')

    Valid properties:
        cliponaxis
            Determines whether or not markers and text nodes are
            clipped about the subplot axes. To show markers and
            text nodes above axis lines and tick labels, make sure
            to set `xaxis.layer` and `yaxis.layer` to *below
            traces*.
        connectgaps
            Determines whether or not gaps (i.e. {nan} or missing
            values) in the provided data arrays are connected.
        customdata
            Assigns extra data each datum. This may be useful when
            listening to hover, click and selection events. Note
            that, "scatter" traces also appends customdata items in
            the markers DOM elements
        customdatasrc
            Sets the source reference on Chart Studio Cloud for
            customdata .
        dx
            Sets the x coordinate step. See 